# Project Overview

This project focuses on predicting the star rating of company reviews using Natural Language Processing (NLP) and Deep Learning techniques. The dataset used for this project was obtained from the Kaggle competition "Sentiment Analysis - Company Reviews: Predict the number of stars given in company reviews". The objective is to build a model that can analyze the textual content of a review and accurately predict its rating on a scale of 1 to 5 stars.

To accomplish this, the review text was preprocessed, tokenized, and converted into numerical sequences. An LSTM-based neural network was then trained to learn patterns in the reviews and predict the corresponding star ratings.

Note: Original 'train dataset' used in 'explanation notebook' is used to split into train-validation-test for checking accuracy of the model

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("C:\\Users\\bhavy\\OneDrive\\Desktop\\Projects\\Review_rating\\review_dataset.csv")

In [3]:
fake_df = pd.read_csv("synthetic_reviews.csv")

In [4]:
df.drop(['Id'],axis=1,inplace=True)

In [5]:
fake_df.drop(['Id'],axis=1,inplace=True)

In [6]:
df = pd.concat([df, fake_df], ignore_index=True)

In [7]:
df.shape

(83342, 2)

In [8]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_t,X_test,y_t,y_test = train_test_split(df['Review'],df['Rating'],stratify=df['Rating'],test_size=0.2,random_state=42)

In [11]:
X_train, X_validation,y_train,y_validation = train_test_split(X_t,y_t,stratify=y_t,test_size=0.2,random_state=42)

In [12]:
X_train.values[1]

'Do not order from deliveroo they are useless order never arrived they took my money'

In [13]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer

In [14]:
tokenizer = Tokenizer(oov_token='New_word')

In [15]:
tokenizer.fit_on_texts(X_train)

In [16]:
tokenizer.word_index

{'New_word': 1,
 'the': 2,
 'and': 3,
 'to': 4,
 'i': 5,
 'a': 6,
 'was': 7,
 'my': 8,
 'it': 9,
 'for': 10,
 'of': 11,
 'they': 12,
 'with': 13,
 'in': 14,
 'on': 15,
 'is': 16,
 'have': 17,
 'service': 18,
 'that': 19,
 'this': 20,
 'not': 21,
 'me': 22,
 'but': 23,
 'as': 24,
 'you': 25,
 'delivery': 26,
 'had': 27,
 'be': 28,
 'very': 29,
 'no': 30,
 'from': 31,
 'so': 32,
 'customer': 33,
 'phone': 34,
 'would': 35,
 'them': 36,
 'time': 37,
 'an': 38,
 'been': 39,
 'at': 40,
 'order': 41,
 'when': 42,
 'are': 43,
 'good': 44,
 'all': 45,
 'will': 46,
 'up': 47,
 'their': 48,
 'get': 49,
 'great': 50,
 'were': 51,
 'which': 52,
 'out': 53,
 'company': 54,
 'after': 55,
 'again': 56,
 'now': 57,
 'we': 58,
 'product': 59,
 'day': 60,
 'what': 61,
 'one': 62,
 'or': 63,
 'just': 64,
 'if': 65,
 'use': 66,
 'told': 67,
 'do': 68,
 'new': 69,
 'then': 70,
 'days': 71,
 'easy': 72,
 'arrived': 73,
 'back': 74,
 'by': 75,
 'received': 76,
 'has': 77,
 'still': 78,
 'ordered': 79,
 'ther

In [17]:
total_vocab_len = len(tokenizer.word_index)

In [18]:
total_vocab_len

31276

In [19]:
int_encoded_X_train = tokenizer.texts_to_sequences(X_train)
int_encoded_X_val   = tokenizer.texts_to_sequences(X_validation)
int_encoded_X_test  = tokenizer.texts_to_sequences(X_test)

In [20]:
int_encoded_X_train[1]

[68, 21, 41, 31, 657, 12, 43, 472, 41, 93, 73, 12, 123, 8, 90]

In [21]:
review_lengths=[]

for review in int_encoded_X_train:
    review_length = len(review)
    review_lengths.append(review_length)

In [22]:
print("Max length is: ",max(review_lengths))
print("90th percentile: ",np.percentile(review_lengths,90))
print("99.5th percentile: ",np.percentile(review_lengths,99.5))

Max length is:  1299
90th percentile:  107.0
99.5th percentile:  437.0


In [23]:
max_length = int(np.percentile(review_lengths,99.5))

In [24]:
from tensorflow.keras.utils import pad_sequences

In [25]:
X_train_padded = pad_sequences(int_encoded_X_train, maxlen=max_length,padding='post',truncating='post')
X_validation_padded = pad_sequences(int_encoded_X_val, maxlen=max_length,padding='post',truncating='post')
X_test_padded = pad_sequences(int_encoded_X_test, maxlen=max_length,padding='post',truncating='post')

In [26]:
X_train_padded[1]

array([ 68,  21,  41,  31, 657,  12,  43, 472,  41,  93,  73,  12, 123,
         8,  90,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [27]:
y_train = y_train - 1

y_train.value_counts().sort_index(ascending=True)

Rating
0    11944
1     6400
2     6400
3     6400
4    22194
Name: count, dtype: int64

In [28]:
y_validation = y_validation - 1

# Creating the Neural network architecture

In [29]:
# tensorflow, keras, sequential, dense

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout

In [30]:
model = Sequential()

model.add(Embedding(input_dim=total_vocab_len+1,output_dim=128,input_shape=(max_length,)))

model.add(LSTM(units=64,dropout=0.2))

model.add(Dense(5,activation='softmax'))

C:\Users\bhavy\anaconda3\envs\pybase-TF2.0\lib\site-packages\keras\src\layers\core\embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [31]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 437, 128)       │     4,003,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,053,189 (15.46 MB)

 Trainable params: 4,053,189 (15.46 MB)

 Non-trainable params: 0 (0.00 B)

# Compiling the network

In [32]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

# Training the network

In [33]:
from tensorflow.keras.callbacks import EarlyStopping

In [34]:
earlystop = EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True)

In [35]:
model.fit(X_train_padded,y_train,epochs=20, batch_size=256,validation_data=(X_validation_padded, y_validation),callbacks=[earlystop])

Epoch 1/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 150s 711ms/step - accuracy: 0.4144 - loss: 1.4786 - val_accuracy: 0.4219 - val_loss: 1.4624
Epoch 2/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 152s 728ms/step - accuracy: 0.4217 - loss: 1.4600 - val_accuracy: 0.4223 - val_loss: 1.4558
Epoch 3/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 153s 732ms/step - accuracy: 0.4214 - loss: 1.4647 - val_accuracy: 0.4219 - val_loss: 1.4577
Epoch 4/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 151s 724ms/step - accuracy: 0.4169 - loss: 1.4632 - val_accuracy: 0.4220 - val_loss: 1.4565
Epoch 5/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 148s 710ms/step - accuracy: 0.4219 - loss: 1.4573 - val_accuracy: 0.4220 - val_loss: 1.4570
Epoch 6/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 149s 715ms/step - accuracy: 0.4215 - loss: 1.4584 - val_accuracy: 0.4220 - val_loss: 1.4562
Epoch 7/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 161s 771ms/step - accuracy: 0.4180 - loss: 1.4598 - val_accuracy: 0.4219 - val_loss: 1.4562
Epoch 8/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 151s 722ms/step - accuracy: 0.4262 -

# Prediction

In [36]:
prob_predictions = model.predict(X_test_padded)

521/521 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step


In [37]:
y_predictions = np.argmax(prob_predictions,axis=1)

In [38]:
y_predictions = y_predictions + 1

In [39]:
y_predictions

array([4, 5, 5, ..., 5, 2, 3])

# Checking metrics

In [40]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_absolute_error

In [41]:
print("Accuracy: ", accuracy_score(y_test,y_predictions))
print("\n")
print("MAE: ", mean_absolute_error(y_test,y_predictions))
print("\n")
print("Confusion matrix: ", confusion_matrix(y_test,y_predictions))
print("\n")
print("Classification report: ", classification_report(y_test,y_predictions))

Accuracy:  0.8358029875817385


MAE:  0.22814805927170195


Confusion matrix:  [[3491   99   45   12   86]
 [ 285 1529  118   25   43]
 [ 118  362 1281  165   74]
 [  45   24  307 1071  553]
 [  81   10   59  226 6560]]


Classification report:                precision    recall  f1-score   support

           1       0.87      0.94      0.90      3733
           2       0.76      0.76      0.76      2000
           3       0.71      0.64      0.67      2000
           4       0.71      0.54      0.61      2000
           5       0.90      0.95      0.92      6936

    accuracy                           0.84     16669
   macro avg       0.79      0.76      0.77     16669
weighted avg       0.83      0.84      0.83     16669



# Saving the model

In [43]:
model.save("classification_review_rating_model.keras")

In [44]:
import pickle

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save max_length
with open("max_length.pkl", "wb") as f:
    pickle.dump(max_length, f)

In [45]:
from tensorflow.keras.models import load_model
model = load_model("classification_review_rating_model.keras")

saving length of each review and saving the tokenizer used